In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import yfinance as yf
import tensorflow as tf
from datetime import datetime, timedelta
import joblib
import warnings
warnings.filterwarnings('ignore')

# Cell 2: Load Model and Scaler
model = tf.keras.models.load_model('forex_analysis_complete.h5')
print("Model loaded successfully")

# Cell 3: Fetch Live Data
def get_live_forex_data(symbol='EURUSD=X', period='60d', interval='1h'):
    """
    Fetch latest Forex data from Yahoo Finance
    """
    print(f"Fetching live data for {symbol}...")
    df = yf.download(symbol, period=period, interval=interval)
    df.columns = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df

# Get latest data
live_df = get_live_forex_data()
print(f"Latest data shape: {live_df.shape}")
print(f"Last 5 rows:\n{live_df.tail()}")

# Cell 4: Calculate Technical Indicators for Live Data
def calculate_indicators_live(df):
    """
    Calculate same indicators as in training
    """
    import pandas_ta as ta
    
    df = df.copy()
    
    # Basic indicators
    df['SMA_20'] = ta.sma(df['Close'], length=20)
    df['EMA_12'] = ta.ema(df['Close'], length=12)
    df['RSI_14'] = ta.rsi(df['Close'], length=14)
    
    # MACD
    macd = ta.macd(df['Close'])
    df['MACD'] = macd['MACD_12_26_9']
    df['MACD_signal'] = macd['MACDs_12_26_9']
    
    # Bollinger Bands
    bbands = ta.bbands(df['Close'])
    df['BB_upper'] = bbands['BBU_20_2.0']
    df['BB_lower'] = bbands['BBL_20_2.0']
    
    # ATR
    df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'])
    
    # ADX
    adx = ta.adx(df['High'], df['Low'], df['Close'])
    df['ADX'] = adx['ADX_14']
    
    return df.dropna()

# Calculate indicators
live_df_indicators = calculate_indicators_live(live_df)
print(f"Data with indicators shape: {live_df_indicators.shape}")
print(f"\nLatest values:\n{live_df_indicators.tail(1)}")

# Cell 5: Prepare Data for Prediction
def prepare_live_sequence(data, window_size=60):
    """
    Prepare the last window of data for prediction
    """
    if len(data) < window_size:
        raise ValueError(f"Need at least {window_size} data points, only have {len(data)}")
    
    # Use last window_size rows
    sequence = data.iloc[-window_size:].values
    sequence = sequence.reshape(1, window_size, -1)
    return sequence

# Get feature columns (should match training)
feature_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'EMA_12', 
                   'RSI_14', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'ATR', 'ADX']

# Ensure all columns exist
available_columns = [col for col in feature_columns if col in live_df_indicators.columns]
print(f"Using {len(available_columns)} features: {available_columns}")

# Prepare sequence
live_sequence = prepare_live_sequence(live_df_indicators[available_columns])
print(f"Sequence shape: {live_sequence.shape}")

# Cell 6: Make Predictions
print("\n" + "="*50)
print("LIVE FOREX ANALYSIS RESULTS")
print("="*50)
print(f"Analysis Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Currency Pair: EUR/USD")
print(f"Latest Price: ${live_df_indicators['Close'].iloc[-1]:.5f}")

# Run prediction
predictions = model.predict(live_sequence)

# Extract predictions
pred_direction = np.argmax(predictions[0], axis=1)[0]
pred_volatility = np.argmax(predictions[1], axis=1)[0]
pred_price_level = (predictions[2] > 0.5).astype(int).flatten()[0]
pred_trend = predictions[3].flatten()[0]

# Cell 7: Interpret Results
print("\n" + "-"*50)
print("ANALYSIS RESULTS")
print("-"*50)

# Direction analysis
direction_map = {0: "📉 DOWN (Bearish)", 1: "📈 UP (Bullish)"}
print(f"\n1. DIRECTION PREDICTION:")
print(f"   → {direction_map[pred_direction]}")
confidence_dir = np.max(predictions[0]) * 100
print(f"   → Confidence: {confidence_dir:.1f}%")

# Volatility analysis
volatility_map = {0: "🟢 LOW Volatility (Ranging Market)", 
                 1: "🟡 MEDIUM Volatility (Normal)",
                 2: "🔴 HIGH Volatility (Risky)"}
print(f"\n2. VOLATILITY REGIME:")
print(f"   → {volatility_map[pred_volatility]}")

# Support/Resistance analysis
price_level_map = {0: "⚪ Normal Range", 
                  1: "⚠️ Near Resistance Level (Potential Reversal)"}
print(f"\n3. PRICE LEVEL:")
print(f"   → {price_level_map[pred_price_level]}")
if pred_price_level == 1:
    print(f"   → Current price: ${live_df_indicators['Close'].iloc[-1]:.5f}")
    print(f"   → Resistance: ${live_df_indicators['BB_upper'].iloc[-1]:.5f}")

# Trend strength analysis
trend_strength = pred_trend * 100  # ADX scale 0-100
print(f"\n4. TREND STRENGTH (ADX):")
print(f"   → ADX Value: {trend_strength:.1f}")
if trend_strength < 20:
    print("   → Weak trend / Ranging market")
elif trend_strength < 40:
    print("   → Moderate trend")
else:
    print("   → Strong trend (Trending market)")

# Cell 8: Generate Trading Signals
print("\n" + "-"*50)
print("TRADING SIGNALS")
print("-"*50)

# Combine signals
signal_score = 0
signal_reasons = []

# Direction signal
if pred_direction == 1:
    signal_score += 1
    signal_reasons.append("Bullish direction")
else:
    signal_score -= 1
    signal_reasons.append("Bearish direction")

# Volatility signal
if pred_volatility == 0:
    signal_reasons.append("Low volatility - Good for range trading")
elif pred_volatility == 2:
    signal_reasons.append("High volatility - Use tighter stops")
    signal_score -= 0.5

# Resistance signal
if pred_price_level == 1:
    signal_reasons.append("Near resistance - Consider taking profits")
    signal_score -= 0.5

# Trend strength signal
if trend_strength > 30:
    signal_reasons.append("Strong trend confirmation")
    signal_score += 0.5

# Final signal
print("\n📊 RECOMMENDATION:")
if signal_score >= 1.5:
    print("   ✅ STRONG BUY / LONG POSITION")
    print("   → Consider entering long with 1-2% risk")
elif signal_score >= 0.5:
    print("   ✅ BUY / LONG POSITION")
    print("   → Cautious long entry with tight stop loss")
elif signal_score >= -0.5:
    print("   ⚠️ NEUTRAL / WAIT")
    print("   → No clear signal, wait for confirmation")
elif signal_score >= -1.5:
    print("   ❌ SELL / SHORT POSITION")
    print("   → Consider short position")
else:
    print("   ❌ STRONG SELL / SHORT POSITION")
    print("   → Aggressive short entry")

print("\n📝 REASONS:")
for reason in signal_reasons:
    print(f"   • {reason}")

# Cell 9: Risk Assessment
print("\n" + "-"*50)
print("RISK ASSESSMENT")
print("-"*50)

# Calculate current volatility based on ATR
current_atr = live_df_indicators['ATR'].iloc[-1]
current_price = live_df_indicators['Close'].iloc[-1]
atr_percentage = (current_atr / current_price) * 100

print(f"Current ATR: ${current_atr:.5f} ({atr_percentage:.2f}% of price)")
print(f"Suggested Stop Loss: {atr_percentage * 1.5:.2f}% away")
print(f"Suggested Take Profit: {atr_percentage * 3:.2f}% away")

# Risk level based on volatility
if atr_percentage > 1.0:
    print("⚠️ HIGH RISK - Reduce position size")
elif atr_percentage > 0.5:
    print("⚖️ MODERATE RISK - Normal position size")
else:
    print("✅ LOW RISK - Can increase position size")

# Cell 10: Save Results
analysis_results = {
    'timestamp': datetime.now(),
    'pair': 'EURUSD',
    'price': current_price,
    'direction': pred_direction,
    'direction_confidence': float(confidence_dir),
    'volatility_regime': pred_volatility,
    'price_level': pred_price_level,
    'trend_strength': float(trend_strength),
    'signal_score': signal_score,
    'atr': float(current_atr),
    'risk_percentage': atr_percentage
}

results_df = pd.DataFrame([analysis_results])
results_df.to_csv('live_analysis_results.csv', index=False)
print("\n✅ Analysis results saved to 'live_analysis_results.csv'")

# Cell 11: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price with Bollinger Bands
axes[0,0].plot(live_df_indicators.index[-100:], live_df_indicators['Close'][-100:], label='Close', linewidth=1.5)
axes[0,0].plot(live_df_indicators.index[-100:], live_df_indicators['BB_upper'][-100:], label='BB Upper', alpha=0.5)
axes[0,0].plot(live_df_indicators.index[-100:], live_df_indicators['BB_lower'][-100:], label='BB Lower', alpha=0.5)
axes[0,0].fill_between(live_df_indicators.index[-100:], live_df_indicators['BB_lower'][-100:], 
                        live_df_indicators['BB_upper'][-100:], alpha=0.1)
axes[0,0].set_title('EURUSD Price with Bollinger Bands')
axes[0,0].set_ylabel('Price')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# RSI
axes[0,1].plot(live_df_indicators.index[-100:], live_df_indicators['RSI_14'][-100:], color='purple')
axes[0,1].axhline(70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[0,1].axhline(30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[0,1].fill_between(live_df_indicators.index[-100:], 30, 70, alpha=0.1)
axes[0,1].set_title('RSI (14)')
axes[0,1].set_ylabel('RSI')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# MACD
axes[1,0].plot(live_df_indicators.index[-100:], live_df_indicators['MACD'][-100:], label='MACD')
axes[1,0].plot(live_df_indicators.index[-100:], live_df_indicators['MACD_signal'][-100:], label='Signal')
axes[1,0].bar(live_df_indicators.index[-100:], live_df_indicators['MACD'] - live_df_indicators['MACD_signal'], 
             alpha=0.3, label='Histogram')
axes[1,0].set_title('MACD')
axes[1,0].set_ylabel('MACD')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# ADX
axes[1,1].plot(live_df_indicators.index[-100:], live_df_indicators['ADX'][-100:], color='red')
axes[1,1].axhline(20, color='g', linestyle='--', alpha=0.5, label='Weak Trend (20)')
axes[1,1].axhline(40, color='orange', linestyle='--', alpha=0.5, label='Strong Trend (40)')
axes[1,1].fill_between(live_df_indicators.index[-100:], 0, 20, alpha=0.1, color='gray')
axes[1,1].fill_between(live_df_indicators.index[-100:], 20, 40, alpha=0.1, color='yellow')
axes[1,1].fill_between(live_df_indicators.index[-100:], 40, 100, alpha=0.1, color='red')
axes[1,1].set_title('ADX - Trend Strength')
axes[1,1].set_ylabel('ADX')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()